In [1]:
# ==============================================================================
# ONE-CELL COLAB SETUP + FULL OOM-SAFE KNOWLEDGE BASE BUILD
# ==============================================================================
# IMPORTANT:
# - Run this in Google Colab, preferably with a GPU runtime.
# - This cell installs packages, mounts Drive, lets you upload local PubMed/PDF
#   files, places PDFs into MyDrive/PDF_Sources, and then runs the full KB build.
# - No forced runtime restart is used, because restarting in the middle of one
#   cell would stop all code below it.
# ==============================================================================

# ------------------------------------------------------------------------------
# A) INSTALL REQUIRED PACKAGES
# ------------------------------------------------------------------------------
!pip install -q pillow==11.1.0 pdfplumber==0.11.4
!pip install -q faiss-cpu sentence-transformers rank-bm25 beautifulsoup4 lxml tqdm requests datasets

# ------------------------------------------------------------------------------
# B) MOUNT GOOGLE DRIVE
# ------------------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# ------------------------------------------------------------------------------
# C) BASIC IMPORTS + PDF FOLDER
# ------------------------------------------------------------------------------
import os
import glob
import shutil
import pdfplumber
from tqdm import tqdm

PDF_FOLDER = '/content/drive/MyDrive/PDF_Sources'
os.makedirs(PDF_FOLDER, exist_ok=True)

print('PDF folder:', PDF_FOLDER)
print('Folder exists:', os.path.exists(PDF_FOLDER))

# ------------------------------------------------------------------------------
# D) UPLOAD LOCAL PUBMED .TXT FILES AND/OR PDF TEXTBOOKS
# ------------------------------------------------------------------------------
# Select the PubMed text files and PDF textbooks from your computer.
# PubMed .txt files stay in /content so the original paths used by the KB script
# continue to work. PDF files are copied automatically into MyDrive/PDF_Sources.
from google.colab import files

print('\nSelect your PubMed (.txt) files and PDF textbooks from your computer:')
uploaded = files.upload()

for filename in uploaded.keys():
    local_path = os.path.join('/content', filename)

    if filename.lower().endswith('.pdf'):
        destination = os.path.join(PDF_FOLDER, filename)
        shutil.copy2(local_path, destination)
        print(f'✅ PDF copied to Drive: {destination}')
    elif filename.lower().endswith('.txt'):
        print(f'✅ PubMed/text file available at: {local_path}')
    else:
        print(f'ℹ️ Uploaded file kept in /content: {local_path}')

print('\nPDFs currently in PDF_Sources:')
pdf_now = sorted(glob.glob(os.path.join(PDF_FOLDER, '*.pdf')))
for p in pdf_now:
    print('  -', os.path.basename(p))
print(f'Total PDFs: {len(pdf_now)}')

# ==============================================================================
# FULL KNOWLEDGE BASE BUILD STARTS BELOW
# ==============================================================================

# ==============================================================================
# FINAL MERGED KNOWLEDGE BASE BUILD
# All 5 Categories + General Medical | Best Sources from Both Versions
# ==============================================================================
#
# CATEGORIES COVERED:
#   - biomedical_engineer
#   - clinical_laboratory_scientist
#   - clinical_psychologist
#   - occupational_therapist
#   - speech_pathologist          ← Uses IMPROVED sources from Code 2
#   - general_medical             ← MedQuAD + MedMCQA
#
# WHAT'S IMPROVED vs Code 1:
#   1. Speech pathologist sources fully replaced with richer Code 2 sources
#      (expanded PubMed terms, Wikipedia, PMC full-text, NIDCD, custom URLs)
#   2. Improved BM25 tokenizer (stopword removal + punctuation stripping)
#   3. Better PDF extractor (tables included, page-level error handling)
#   4. Better web scraper (retry logic, more content selectors)
#   5. Consistent 200-word chunk size for ALL sources
#
# INSTRUCTIONS:
#   1. Upload your PubMed flat files to Colab (paths in PUBMED_FILES below)
#   2. Upload your PDF textbooks to Google Drive: "My Drive/PDF_Sources/"
#      - Book_19.pdf, Book_23.pdf, Book_25.pdf, book 8.pdf, book1.pdf,
#        book4.pdf, book_14.pdf, book_15.pdf, book_17.pdf, book_18.pdf
#        celf-5-clinical-evaluation-of-language-fundamentals-5th-edition.pdf
#   3. Run the entire script in Google Colab (GPU runtime recommended)
# ==============================================================================

import os
# Reduce CUDA allocator fragmentation before SentenceTransformer/PyTorch use CUDA.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import re
import time
import pickle
import hashlib
import glob
import warnings
import numpy as np
import pandas as pd
import faiss
import requests
import xml.etree.ElementTree as ET
import pdfplumber
from tqdm import tqdm
from bs4 import BeautifulSoup
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from google.colab import drive

warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------
# 1) GOOGLE DRIVE CONFIG (already mounted above)
# -----------------------------------------------------------------------
SAVE_PATH   = '/content/drive/MyDrive/KB_complete'
PDF_FOLDER  = '/content/drive/MyDrive/PDF_Sources'
os.makedirs(SAVE_PATH, exist_ok=True)
os.makedirs(PDF_FOLDER, exist_ok=True)
print(f"Save path : {SAVE_PATH}")
print(f"PDF folder: {PDF_FOLDER}")

# Embedding model
EMBED_MODEL = 'BAAI/bge-base-en-v1.5'

# Chunking — consistent for ALL sources
CHUNK_SIZE    = 200
CHUNK_OVERLAP = 50
MIN_CHUNK_WORDS = 25

# All categories
CATEGORIES = [
    'biomedical_engineer',
    'clinical_laboratory_scientist',
    'clinical_psychologist',
    'occupational_therapist',
    'speech_pathologist',
    'general_medical',
]

# PubMed flat files — adjust paths if needed
PUBMED_FILES = {
    'clinical_laboratory_scientist': '/content/pubmed-ClinicalLa-set.txt',
    'clinical_psychologist':         '/content/pubmed-ClinicalPs-set.txt',
    'occupational_therapist':        '/content/pubmed-Occupation-set.txt',
    'biomedical_engineer':           '/content/pubmed-biomedical-set (1).txt',
    'speech_pathologist':            '/content/pubmed-speechlang-set.txt',
}

print(f"\nEmbed model : {EMBED_MODEL}")
print(f"Chunk size  : {CHUNK_SIZE} words  |  overlap: {CHUNK_OVERLAP} words")
print("\nPubMed files:")
for cat, path in PUBMED_FILES.items():
    status = "✅" if os.path.exists(path) else "❌ NOT FOUND"
    print(f"  {status}  {cat:<40}  {path}")


# -----------------------------------------------------------------------
# 2) UTILITY FUNCTIONS  (all from Code 2 — best versions)
# -----------------------------------------------------------------------

def clean_text(text):
    """Normalize whitespace and strip non-ASCII."""
    if not text:
        return ''
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\x20-\x7E]', '', text)
    return text.strip()


def chunk_with_overlap(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Split text into overlapping word-count chunks."""
    words = text.split()
    if not words:
        return []
    step = max(1, chunk_size - overlap)
    chunks = []
    for start in range(0, len(words), step):
        chunk = ' '.join(words[start:start + chunk_size])
        if len(chunk.split()) >= MIN_CHUNK_WORDS:
            chunks.append(chunk)
        if start + chunk_size >= len(words):
            break
    return chunks


# Improved BM25 tokenizer (Code 2)
STOPWORDS = frozenset({
    'the','a','an','is','are','was','were','be','been','being',
    'have','has','had','do','does','did','will','would','shall',
    'should','may','might','can','could','and','but','or','nor',
    'not','no','so','if','then','than','that','this','these',
    'those','it','its','of','in','on','at','to','for','with',
    'by','from','as','into','about','between','through','after',
    'before','during','without','within','also','more','most',
    'very','such','each','both','all','any','some','other',
    'which','what','when','where','how','who','whom','whose',
    'there','here','just','only','own','same','our','your',
    'their','my','his','her','we','they','you','he','she',
    'me','him','us','them','i','am',
})

def tokenize_for_bm25(text):
    """Tokenize with punctuation removal and stopword filtering."""
    tokens = re.findall(r'[a-z0-9]+', text.lower())
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]


# Dedup + storage
SEEN_HASHES = set()
ALL_CHUNKS  = []

def add_chunks(chunks, category, source):
    """Add chunks with MD5 deduplication."""
    added = 0
    for c in chunks:
        h = hashlib.md5(c.encode()).hexdigest()
        if h not in SEEN_HASHES:
            SEEN_HASHES.add(h)
            ALL_CHUNKS.append({'category': category, 'source': source, 'content': c})
            added += 1
    return added


# Improved web scraper (Code 2 — retry + broader selectors)
def scrape_url(url, min_len=300, retries=2):
    headers = {
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/120.0.0.0 Safari/537.36'
        )
    }
    for attempt in range(retries + 1):
        try:
            r = requests.get(url, headers=headers, timeout=15)
            r.raise_for_status()
            soup = BeautifulSoup(r.text, 'html.parser')
            main = (
                soup.find('div', id='topic-summary')        or
                soup.find('div', class_='article-body')     or
                soup.find('div', id='mc')                   or
                soup.find('div', class_='jig-ncbiinpagenav') or
                soup.find('article')                        or
                soup.find('div', class_='main-content')     or
                soup.find('main')                           or
                soup.body                                   or
                soup
            )
            for tag in main.find_all(['script','style','nav','footer',
                                       'header','aside','button','form','iframe']):
                tag.decompose()
            text = clean_text(main.get_text(separator=' '))
            if len(text) >= min_len:
                return text
        except Exception:
            if attempt < retries:
                time.sleep(1)
    return ''


# Improved PDF extractor (Code 2 — includes tables)
def extract_text_from_pdf(pdf_path):
    all_text = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                try:
                    page_text = page.extract_text()
                    if page_text and len(page_text.strip()) > 30:
                        all_text.append(page_text)
                    for table in page.extract_tables():
                        for row in table:
                            if row:
                                row_text = ' '.join(str(cell) for cell in row if cell)
                                if len(row_text.strip()) > 10:
                                    all_text.append(row_text)
                except Exception:
                    continue
        return clean_text(' '.join(all_text))
    except Exception as e:
        print(f"    PDF error [{os.path.basename(pdf_path)}]: {str(e)[:80]}")
        return ''


print("\n✅ All utility functions ready.")


# -----------------------------------------------------------------------
# 3) SOURCE 1 — LOCAL PUBMED FLAT FILES  (all 5 categories)
# -----------------------------------------------------------------------
print("\n" + "="*80)
print("SOURCE 1: LOCAL PUBMED FLAT FILES")
print("="*80)

def parse_pubmed_flat_file(file_path):
    if not os.path.exists(file_path):
        print(f'  ⚠️  Not found: {file_path}')
        return []
    with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
        content = f.read()
    articles = []
    for record in re.split(r'\nPMID- ', content):
        if not record.strip():
            continue
        pmid_m     = re.match(r'^(\d+)', record.strip())
        title_m    = re.search(r'\nTI  - (.*?)(?=\n[A-Z]{2,4}  -|\Z)', record, re.DOTALL)
        abstract_m = re.search(r'\nAB  - (.*?)(?=\n[A-Z]{2,4}  -|\Z)', record, re.DOTALL)
        pmid     = pmid_m.group(1)     if pmid_m     else 'unknown'
        title    = clean_text(title_m.group(1))    if title_m    else ''
        abstract = clean_text(abstract_m.group(1)) if abstract_m else ''
        if abstract and len(abstract) > 50:
            articles.append({'pmid': pmid, 'title': title, 'abstract': abstract})
    return articles

for cat, fpath in PUBMED_FILES.items():
    arts = parse_pubmed_flat_file(fpath)
    n = 0
    for art in arts:
        full = f"Title: {art['title']}. {art['abstract']}"
        n += add_chunks(chunk_with_overlap(clean_text(full)), cat, 'pubmed_file')
    print(f'  {cat:<42}  {len(arts):>5} articles  →  {n} chunks')

print(f'\nRUNNING TOTAL: {len(ALL_CHUNKS):,}')


# -----------------------------------------------------------------------
# 4) SOURCE 2 — PUBMED API
#    Non-speech categories: Code 1 terms
#    Speech pathologist   : Code 2 expanded terms
# -----------------------------------------------------------------------
print("\n" + "="*80)
print("SOURCE 2: PUBMED API")
print("="*80)

def fetch_pubmed_api(term, max_results=50):
    base = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils'
    try:
        ids = requests.get(f'{base}/esearch.fcgi',
            params={'db':'pubmed','term':term,'retmax':max_results,
                    'retmode':'json','datetype':'pdat',
                    'mindate':'2015','maxdate':'2025'},
            timeout=15).json()['esearchresult']['idlist']
    except:
        return []
    if not ids:
        return []
    try:
        xml = requests.get(f'{base}/efetch.fcgi',
            params={'db':'pubmed','id':','.join(ids[:50]),
                    'rettype':'abstract','retmode':'xml'},
            timeout=30).text
    except:
        return []
    time.sleep(0.4)
    arts = []
    try:
        root = ET.fromstring(xml)
        for el in root.findall('.//PubmedArticle'):
            pmid_el      = el.find('.//PMID')
            title_el     = el.find('.//ArticleTitle')
            abstract_els = el.findall('.//AbstractText')
            pmid     = pmid_el.text if pmid_el is not None else 'unknown'
            title    = clean_text(title_el.text or '') if title_el is not None else ''
            abstract = clean_text(' '.join(a.text or '' for a in abstract_els))
            if abstract and len(abstract) > 50:
                arts.append({'pmid': pmid, 'title': title, 'abstract': abstract})
    except:
        pass
    return arts

API_SEARCHES = {
    # ── Non-speech: Code 1 terms ──────────────────────────────────────
    'biomedical_engineer': [
        'medical device', 'neural engineering', 'tissue engineering',
    ],
    'clinical_laboratory_scientist': [
        'clinical laboratory', 'blood count',
    ],
    'clinical_psychologist': [
        'cognitive behavioral therapy', 'depression',
    ],
    'occupational_therapist': [
        'occupational therapy', 'activities daily living',
    ],

    # ── Speech pathologist: Code 2 expanded terms ─────────────────────
    'speech_pathologist': [
        # Aphasia
        'aphasia therapy', 'aphasia rehabilitation',
        'Broca aphasia treatment', 'Wernicke aphasia',
        'primary progressive aphasia',
        # Motor speech
        'dysarthria treatment', 'childhood apraxia of speech',
        'apraxia of speech assessment',
        # Language
        'developmental language disorder', 'speech sound disorder',
        'language assessment children',
        # Fluency
        'stuttering therapy', 'stuttering severity instrument',
        'stuttering assessment adults',
        # Voice
        'voice disorders treatment', 'voice therapy techniques',
        'vocal nodules treatment', 'spasmodic dysphonia',
        'voice handicap index', 'CAPE-V voice assessment',
        'laryngopharyngeal reflux voice',
        # Swallowing
        'dysphagia rehabilitation', 'swallowing disorders assessment',
        'pharyngeal phase swallowing',
        # Hearing
        'conductive hearing loss', 'sensorineural hearing loss',
        'auditory training', 'central auditory processing disorder',
        # Cognitive & AAC
        'cognitive communication disorder',
        'traumatic brain injury communication',
        'augmentative alternative communication',
        'PECS autism communication',
        'social communication disorder',
        'autism communication intervention',
        # Assessment / treatment methods
        'melodic intonation therapy', 'dynamic assessment language',
    ],
}

seen_pmids = set()
for cat, terms in API_SEARCHES.items():
    n = 0
    for term in tqdm(terms, desc=cat[:30], leave=False):
        for art in fetch_pubmed_api(term, max_results=40):
            if art['pmid'] in seen_pmids:
                continue
            seen_pmids.add(art['pmid'])
            full = f"Title: {art['title']}. {art['abstract']}"
            n += add_chunks(chunk_with_overlap(clean_text(full)), cat, 'pubmed_api')
    print(f'  {cat:<42}  +{n} chunks')

print(f'\nRUNNING TOTAL: {len(ALL_CHUNKS):,}')


# -----------------------------------------------------------------------
# 5) SOURCE 3 — MEDLINEPLUS
#    Non-speech: Code 1 URLs  |  Speech: Code 2 expanded URLs
# -----------------------------------------------------------------------
print("\n" + "="*80)
print("SOURCE 3: MEDLINEPLUS")
print("="*80)

MEDLINEPLUS_PAGES = {
    'biomedical_engineer': [
        'https://medlineplus.gov/mriscans.html',
        'https://medlineplus.gov/medicaldevices.html',
    ],
    'clinical_laboratory_scientist': [
        'https://medlineplus.gov/laboratorytests.html',
        'https://medlineplus.gov/bloodcount.html',
    ],
    'clinical_psychologist': [
        'https://medlineplus.gov/anxiety.html',
        'https://medlineplus.gov/depression.html',
    ],
    'occupational_therapist': [
        'https://medlineplus.gov/rehabilitation.html',
        'https://medlineplus.gov/stroke.html',
    ],
    # Speech: Code 2 expanded list
    'speech_pathologist': [
        'https://medlineplus.gov/speechandcommunicationdisorders.html',
        'https://medlineplus.gov/aphasia.html',
        'https://medlineplus.gov/stuttering.html',
        'https://medlineplus.gov/voicedisorders.html',
        'https://medlineplus.gov/swallowingdisorders.html',
        'https://medlineplus.gov/hearingdisordersanddeafness.html',
    ],
}

for cat, urls in MEDLINEPLUS_PAGES.items():
    n = 0
    for url in tqdm(urls, desc=cat[:30], leave=False):
        text = scrape_url(url)
        time.sleep(0.5)
        if text:
            n += add_chunks(chunk_with_overlap(text), cat, 'medlineplus')
    print(f'  {cat:<42}  +{n} chunks')

print(f'\nRUNNING TOTAL: {len(ALL_CHUNKS):,}')


# -----------------------------------------------------------------------
# 6) SOURCE 4 — NIDCD  (speech_pathologist only — Code 2 expanded)
# -----------------------------------------------------------------------
print("\n" + "="*80)
print("SOURCE 4: NIDCD (NIH Speech & Hearing)")
print("="*80)

NIDCD_URLS = [
    'https://www.nidcd.nih.gov/health/aphasia',
    'https://www.nidcd.nih.gov/health/dysarthria',
    'https://www.nidcd.nih.gov/health/developmental-language-disorder',
    'https://www.nidcd.nih.gov/health/apraxia-speech',
    'https://www.nidcd.nih.gov/health/stuttering',
    'https://www.nidcd.nih.gov/health/voice-speech-and-language',
]

n = 0
for url in tqdm(NIDCD_URLS, desc='NIDCD'):
    text = scrape_url(url)
    time.sleep(0.5)
    if text:
        n += add_chunks(chunk_with_overlap(text), 'speech_pathologist', 'nidcd')
print(f'  speech_pathologist  +{n} chunks')
print(f'\nRUNNING TOTAL: {len(ALL_CHUNKS):,}')


# -----------------------------------------------------------------------
# 7) SOURCE 5 — NIMH  (clinical_psychologist — Code 1)
# -----------------------------------------------------------------------
print("\n" + "="*80)
print("SOURCE 5: NIMH (NIH Mental Health)")
print("="*80)

NIMH_URLS = [
    'https://www.nimh.nih.gov/health/topics/depression',
    'https://www.nimh.nih.gov/health/topics/anxiety-disorders',
    'https://www.nimh.nih.gov/health/topics/bipolar-disorder',
]

n = 0
for url in tqdm(NIMH_URLS, desc='NIMH'):
    text = scrape_url(url)
    time.sleep(0.5)
    if text:
        n += add_chunks(chunk_with_overlap(text), 'clinical_psychologist', 'nimh')
print(f'  clinical_psychologist  +{n} chunks')
print(f'\nRUNNING TOTAL: {len(ALL_CHUNKS):,}')


# -----------------------------------------------------------------------
# 8) SOURCE 6 — CDC  (Code 1 for non-speech; Code 2 URLs for speech)
# -----------------------------------------------------------------------
print("\n" + "="*80)
print("SOURCE 6: CDC")
print("="*80)

CDC_PAGES = {
    'clinical_psychologist': [
        'https://www.cdc.gov/mentalhealth/index.htm',
    ],
    'occupational_therapist': [
        'https://www.cdc.gov/traumaticbraininjury/index.html',
        'https://www.cdc.gov/stroke/index.htm',
    ],
    # Speech: Code 2 URLs
    'speech_pathologist': [
        'https://www.cdc.gov/ncbddd/autism/index.html',
        'https://www.cdc.gov/autism/signs-symptoms/index.html',
    ],
}

for cat, urls in CDC_PAGES.items():
    n = 0
    for url in urls:
        text = scrape_url(url)
        time.sleep(0.8)
        if text:
            n += add_chunks(chunk_with_overlap(text), cat, 'cdc')
    print(f'  {cat:<42}  +{n} chunks')

print(f'\nRUNNING TOTAL: {len(ALL_CHUNKS):,}')


# -----------------------------------------------------------------------
# 9) SOURCE 7 — WHO  (biomedical_engineer + clinical_psychologist — Code 1)
# -----------------------------------------------------------------------
print("\n" + "="*80)
print("SOURCE 7: WHO Fact Sheets")
print("="*80)

WHO_PAGES = {
    'clinical_psychologist': [
        'https://www.who.int/news-room/fact-sheets/detail/depression',
    ],
    'biomedical_engineer': [
        'https://www.who.int/news-room/fact-sheets/detail/cardiovascular-diseases-(cvds)',
    ],
}

for cat, urls in WHO_PAGES.items():
    n = 0
    for url in urls:
        text = scrape_url(url)
        time.sleep(1)
        if text:
            n += add_chunks(chunk_with_overlap(text), cat, 'who')
    print(f'  {cat:<42}  +{n} chunks')

print(f'\nRUNNING TOTAL: {len(ALL_CHUNKS):,}')


# -----------------------------------------------------------------------
# 10) SOURCE 8 — NCBI BOOKSHELF
#     Non-speech: Code 1 URLs (biomedical_engineer)
#     Speech:     Code 2 custom speech pathology URLs
# -----------------------------------------------------------------------
print("\n" + "="*80)
print("SOURCE 8: NCBI Bookshelf")
print("="*80)

# Non-speech NCBI books (Code 1 — biomedical_engineer)
NCBI_BIOMEDICAL_URLS = [
    ('biomedical_engineer', 'https://www.ncbi.nlm.nih.gov/books/NBK554776/'),
    ('biomedical_engineer', 'https://www.ncbi.nlm.nih.gov/books/NBK459455/'),
    ('biomedical_engineer', 'https://www.ncbi.nlm.nih.gov/books/NBK546152/'),
    ('biomedical_engineer', 'https://www.ncbi.nlm.nih.gov/books/NBK482487/'),
    ('biomedical_engineer', 'https://www.ncbi.nlm.nih.gov/books/NBK551686/'),
]

# Speech pathology NCBI/PMC URLs (Code 2)
NCBI_SPEECH_URLS = [
    # Aphasia
    'https://www.ncbi.nlm.nih.gov/books/NBK559315/',
    'https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10614176/',
    # Language Disorders
    'https://www.ncbi.nlm.nih.gov/books/NBK356270/',
    # Dysarthria
    'https://www.ncbi.nlm.nih.gov/books/NBK592453/',
    # Apraxia
    'https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6494637/',
    'https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3816031/',
    # Stuttering
    'https://www.ncbi.nlm.nih.gov/books/NBK603738/',
    'https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7118465/',
    'https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6802914/',
    # Dysphagia
    'https://www.ncbi.nlm.nih.gov/books/NBK559174/',
    'https://www.ncbi.nlm.nih.gov/books/NBK470455/',
    'https://www.ncbi.nlm.nih.gov/books/NBK560734/',
    'https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4953294/',
    # Voice
    'https://www.ncbi.nlm.nih.gov/books/NBK563182/',
    'https://www.ncbi.nlm.nih.gov/books/NBK565881/',
    'https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3775647/',
    'https://www.ncbi.nlm.nih.gov/pmc/articles/PMC9445758/',
    'https://www.ncbi.nlm.nih.gov/pmc/articles/PMC11228133/',
    # Hearing
    'https://www.ncbi.nlm.nih.gov/books/NBK536521/',
    'https://www.ncbi.nlm.nih.gov/books/NBK279600/',
    'https://www.ncbi.nlm.nih.gov/books/NBK536534/',
    'https://www.ncbi.nlm.nih.gov/books/NBK470177/',
    # Cognitive Communication
    'https://www.ncbi.nlm.nih.gov/books/NBK482481/',
    # Autism / Social
    'https://www.ncbi.nlm.nih.gov/books/NBK573609/',
    'https://www.ncbi.nlm.nih.gov/books/NBK430901/',
    # Resonance
    'https://www.ncbi.nlm.nih.gov/books/NBK563128/',
    'https://www.ncbi.nlm.nih.gov/books/NBK563149/',
]

n_bio = 0
for cat, url in tqdm(NCBI_BIOMEDICAL_URLS, desc='NCBI Biomedical'):
    text = scrape_url(url, min_len=500)
    time.sleep(1)
    if text:
        n_bio += add_chunks(chunk_with_overlap(text), cat, 'ncbi_bookshelf')
print(f'  biomedical_engineer  +{n_bio} chunks')

n_slp = 0
for url in tqdm(NCBI_SPEECH_URLS, desc='NCBI Speech'):
    text = scrape_url(url, min_len=300)
    time.sleep(1)
    if text:
        n_slp += add_chunks(chunk_with_overlap(text), 'speech_pathologist', 'ncbi_bookshelf')
print(f'  speech_pathologist   +{n_slp} chunks')

print(f'\nRUNNING TOTAL: {len(ALL_CHUNKS):,}')


# -----------------------------------------------------------------------
# 11) SOURCE 9 — ASHA PDFs  (speech_pathologist — Code 1)
# -----------------------------------------------------------------------
print("\n" + "="*80)
print("SOURCE 9: ASHA PDFs")
print("="*80)

ASHA_PDFS = {
    'slp_medical_review.pdf':    'https://www.asha.org/siteassets/uploadedfiles/slp-medical-review-guidelines.pdf',
    'slp_scope_of_practice.pdf': 'https://www.asha.org/siteassets/publications/sp2016-00343.pdf',
    'slp_adult_healthcare.pdf':  'https://www.asha.org/siteassets/uploadedfiles/adultslpservicesinhealthcare.pdf',
    'slpa_scope.pdf':            'https://www.asha.org/siteassets/publications/slpa-scope-of-practice-2022.pdf',
}

n = 0
for fname, url in ASHA_PDFS.items():
    try:
        fpath = f'/tmp/{fname}'
        r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=30)
        if r.status_code == 200:
            with open(fpath, 'wb') as f:
                f.write(r.content)
            text = extract_text_from_pdf(fpath)   # Using Code 2's improved extractor
            if len(text.split()) >= MIN_CHUNK_WORDS:
                added = add_chunks(chunk_with_overlap(text), 'speech_pathologist', 'asha_pdf')
                n += added
                print(f'  ✅ {fname}  →  {added} chunks')
            os.remove(fpath)
        else:
            print(f'  ⚠️  HTTP {r.status_code}: {fname}')
    except Exception as e:
        print(f'  ⚠️  {fname}: {str(e)[:60]}')

print(f'  speech_pathologist  +{n} chunks')
print(f'\nRUNNING TOTAL: {len(ALL_CHUNKS):,}')


# -----------------------------------------------------------------------
# 12) SOURCE 10 — PMC FULL-TEXT  (speech_pathologist — Code 2)
# -----------------------------------------------------------------------
print("\n" + "="*80)
print("SOURCE 10: PMC Full-Text (Speech Pathologist)")
print("="*80)

def fetch_pmc_articles(term, max_results=20):
    base = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils'
    try:
        ids = requests.get(f'{base}/esearch.fcgi',
            params={'db':'pmc','term':term,'retmax':max_results,'retmode':'json'},
            timeout=20).json()['esearchresult']['idlist']
    except:
        return []
    articles = []
    for pmcid in ids:
        try:
            xml = requests.get(f'{base}/efetch.fcgi',
                params={'db':'pmc','id':pmcid,'retmode':'xml'}, timeout=30).text
            root = ET.fromstring(xml)
            text_parts = [elem.text for elem in root.iter() if elem.text]
            full_text = clean_text(' '.join(text_parts))
            if len(full_text) > 1500:
                articles.append(full_text)
            time.sleep(0.3)
        except:
            pass
    return articles

PMC_SEARCHES = [
    'aphasia', 'primary progressive aphasia',
    'developmental language disorder', 'speech sound disorder',
    'childhood apraxia of speech', 'apraxia of speech',
    'dysarthria', 'stuttering', 'fluency disorders',
    'voice disorders', 'dysphonia',
    'dysphagia', 'swallowing disorders',
    'hearing loss', 'audiology',
    'central auditory processing disorder',
    'autism social communication disorder',
    'social communication disorder',
    'cognitive communication disorders',
    'augmentative and alternative communication',
    'resonance disorders', 'velopharyngeal insufficiency',
    'melodic intonation therapy',
    'voice hygiene laryngopharyngeal reflux',
    'CAPE-V voice evaluation',
    'stuttering severity instrument',
    'total laryngectomy voice rehabilitation',
    'tracheoesophageal speech',
]

n = 0
for term in tqdm(PMC_SEARCHES, desc='PMC Articles'):
    for article in fetch_pmc_articles(term, max_results=15):
        n += add_chunks(chunk_with_overlap(article), 'speech_pathologist', 'pmc_fulltext')
print(f'  speech_pathologist  +{n} chunks')
print(f'\nRUNNING TOTAL: {len(ALL_CHUNKS):,}')


# -----------------------------------------------------------------------
# 13) SOURCE 11 — WIKIPEDIA  (speech_pathologist — Code 2)
# -----------------------------------------------------------------------
print("\n" + "="*80)
print("SOURCE 11: Wikipedia (Speech Pathology — textbook-level definitions)")
print("="*80)

WIKI_TOPICS = [
    'Aphasia', 'Broca%27s_aphasia', 'Wernicke%27s_aphasia',
    'Primary_progressive_aphasia', 'Anomic_aphasia',
    'Dysarthria', 'Apraxia_of_speech',
    'Stuttering', 'Cluttering_(speech)',
    'Dysphagia', 'Swallowing',
    'Voice_disorder', 'Vocal_cord_paresis',
    'Spasmodic_dysphonia', 'Vocal_cord_nodule',
    'Laryngectomy', 'Tracheoesophageal_puncture',
    'Conductive_hearing_loss', 'Sensorineural_hearing_loss',
    'Otosclerosis', 'Auditory_processing_disorder',
    'Melodic_intonation_therapy',
    'Picture_Exchange_Communication_System',
    'Augmentative_and_alternative_communication',
    'Traumatic_brain_injury',
    'Phonation', 'Vocal_folds',
    'Thalamus', 'Broca%27s_area',
    'Zone_of_proximal_development',
    'Velopharyngeal_insufficiency',
    'Cleft_palate',
    'Autism_spectrum_disorder',
    'Anterograde_amnesia',
    'Vital_capacity', 'Pulmonary_function_testing',
    'Hyoid_bone', 'Epiglottis',
    'Laryngopharyngeal_reflux',
    'Bernoulli%27s_principle',
]

n = 0
for topic in tqdm(WIKI_TOPICS, desc='Wikipedia'):
    url = f'https://en.wikipedia.org/wiki/{topic}'
    try:
        text = scrape_url(url, min_len=300)
        time.sleep(0.3)
        if text:
            n += add_chunks(chunk_with_overlap(text), 'speech_pathologist', 'wikipedia')
    except Exception:
        pass
print(f'  speech_pathologist  +{n} chunks')
print(f'\nRUNNING TOTAL: {len(ALL_CHUNKS):,}')


# -----------------------------------------------------------------------
# 14) SOURCE 12 — LOCAL PDF TEXTBOOKS  (all speech_pathologist — Code 2)
# -----------------------------------------------------------------------
print("\n" + "="*80)
print("SOURCE 12: LOCAL PDF TEXTBOOKS")
print("="*80)

pdf_files = []
if os.path.exists(PDF_FOLDER):
    found = sorted(glob.glob(os.path.join(PDF_FOLDER, '*.pdf')))
    found += sorted(glob.glob(os.path.join(PDF_FOLDER, '**', '*.pdf'), recursive=True))
    pdf_files = list(dict.fromkeys(found))
    print(f"  Found {len(pdf_files)} PDFs in: {PDF_FOLDER}")
else:
    print(f"  ⚠️  PDF folder not found: {PDF_FOLDER}")
    print(f"  ⚠️  Create it in Google Drive and upload your textbooks, then re-run.")

n_total_pdf = 0
n_failed    = 0
for pdf_path in tqdm(pdf_files, desc='PDFs'):
    filename = os.path.basename(pdf_path)
    text = extract_text_from_pdf(pdf_path)
    if text and len(text.split()) >= 50:
        chunks = chunk_with_overlap(text)
        added  = add_chunks(chunks, 'speech_pathologist', f'pdf:{filename}')
        n_total_pdf += added
        print(f'  ✅ {filename:<55}  {len(text.split()):>6} words  →  {added} chunks')
    else:
        n_failed += 1
        print(f'  ❌ {filename:<55}  FAILED (no text)')

print(f'\n  PDFs: {len(pdf_files)} total | {len(pdf_files)-n_failed} ok | {n_failed} failed | {n_total_pdf} chunks')
print(f'\nRUNNING TOTAL: {len(ALL_CHUNKS):,}')


# -----------------------------------------------------------------------
# 15) SOURCE 13 — MEDQUAD  (general_medical — Code 1)
# -----------------------------------------------------------------------
print("\n" + "="*80)
print("SOURCE 13: MedQuAD (47K QA pairs)")
print("="*80)

os.system('git clone https://github.com/abachaa/MedQuAD.git >/dev/null 2>&1')

n_medquad = 0
for root_dir, dirs, files in os.walk('MedQuAD'):
    for file in files:
        if not file.endswith('.xml'):
            continue
        path = os.path.join(root_dir, file)
        try:
            tree     = ET.parse(path)
            xml_root = tree.getroot()
            focus_el = xml_root.find('Focus')
            focus    = focus_el.text.strip() if focus_el is not None and focus_el.text else 'General'
            for qa in xml_root.findall('.//QAPair'):
                q_el = qa.find('Question')
                a_el = qa.find('Answer')
                if not (q_el is not None and a_el is not None
                        and q_el.text and a_el.text):
                    continue
                answer = a_el.text.strip()
                if len(answer) < 50:
                    continue
                text = f"Q: {q_el.text.strip()}\nA: {answer}\nTopic: {focus}"
                n_medquad += add_chunks(
                    chunk_with_overlap(clean_text(text)), 'general_medical', 'medquad')
        except Exception:
            pass

print(f'  general_medical  +{n_medquad:,} chunks')
print(f'\nRUNNING TOTAL: {len(ALL_CHUNKS):,}')


# -----------------------------------------------------------------------
# 16) SOURCE 14 — MEDMCQA  (general_medical — Code 1)
# -----------------------------------------------------------------------
print("\n" + "="*80)
print("SOURCE 14: MedMCQA (182K medical explanations)")
print("="*80)

try:
    from datasets import load_dataset
    dataset = load_dataset('openlifescienceai/medmcqa', split='train')
    print(f"  Loaded {len(dataset):,} rows")

    n_mcqa    = 0
    n_skipped = 0
    for row in tqdm(dataset, desc='MedMCQA', leave=False):
        if not row['exp'] or len(row['exp'].strip()) < 30:
            n_skipped += 1
            continue
        options  = {1: row.get('opa',''), 2: row.get('opb',''),
                    3: row.get('opc',''), 4: row.get('opd','')}
        correct  = options.get(row.get('cop', 0), '')
        topic    = row.get('topic_name', 'General Medicine')
        subject  = row.get('subject_name', '')
        text = (
            f"Medical Topic: {topic} ({subject})\n"
            f"Explanation: {row['exp'].strip()}\n"
            f"Correct Fact: {correct}\n"
            f"Source: MedMCQA"
        )
        n_mcqa += add_chunks(
            chunk_with_overlap(clean_text(text)), 'general_medical', 'medmcqa')

    print(f'  general_medical  +{n_mcqa:,} chunks')
    print(f'  Skipped (no explanation): {n_skipped:,}')
except Exception as e:
    print(f'  ⚠️  MedMCQA failed: {e}  — skipping.')

print(f'\nRUNNING TOTAL: {len(ALL_CHUNKS):,}')


# -----------------------------------------------------------------------
# 17) BUILD DATAFRAME
# -----------------------------------------------------------------------
print("\n" + "="*80)
print("BUILDING DATAFRAME")
print("="*80)

df = pd.DataFrame(ALL_CHUNKS).reset_index(drop=True)
print(f'\nTOTAL CHUNKS : {len(df):,}')
print('\nChunks per category:')
print(df['category'].value_counts().to_string())
print('\nChunks per source:')
print(df['source'].value_counts().to_string())


# -----------------------------------------------------------------------
# 18) GENERATE EMBEDDINGS  (OOM-safe incremental version)
# -----------------------------------------------------------------------
print("\n" + "="*80)
print("GENERATING EMBEDDINGS  (OOM-safe incremental mode)")
print("="*80)

import gc
import shutil
import torch

# IMPORTANT:
# PYTORCH_CUDA_ALLOC_CONF is also set near the imports at the top of this
# full script, before SentenceTransformer touches CUDA. Setting it here again
# is harmless and makes this section safe if it is run separately.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Do not alter/reorder the dataframe. We only read content in its existing order.
df = df.reset_index(drop=True)
texts = df['content'].astype(str).tolist()
total = len(texts)

if total == 0:
    raise RuntimeError("No chunks found in df. Sections 1-17 must finish first.")

required_cols = {'category', 'source', 'content'}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise RuntimeError(f"Missing required dataframe columns: {missing_cols}")

# Same model as the original code.
EMBED_MODEL = 'BAAI/bge-base-en-v1.5'

# 512 caused the OOM. Start at 64 and automatically reduce if necessary.
GPU_BATCH_SIZES = [64, 32, 16, 8]
CPU_BATCH_SIZE = 32
SUB_BATCH_SIZE = 50_000
SHARD_DIR = '/content/emb_shards'
os.makedirs(SHARD_DIR, exist_ok=True)

# Remove stale temporary shard files from an interrupted previous attempt.
for old_file in glob.glob(os.path.join(SHARD_DIR, 'shard_*.npy')):
    try:
        os.remove(old_file)
    except Exception:
        pass

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device        : {device}")
print(f"Total chunks  : {total:,}")
print(f"Shard size    : {SUB_BATCH_SIZE:,}")
print(f"Shards needed : {(total + SUB_BATCH_SIZE - 1) // SUB_BATCH_SIZE}")

if torch.cuda.is_available():
    gc.collect()
    torch.cuda.empty_cache()
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"GPU total     : {total_bytes / (1024**3):.2f} GiB")
    print(f"GPU free now  : {free_bytes / (1024**3):.2f} GiB")

print(f"\nLoading model: {EMBED_MODEL}")
embedder = SentenceTransformer(EMBED_MODEL, device=device)
embedding_dim = embedder.get_sentence_embedding_dimension()
print(f"Embedding dim : {embedding_dim}")


def encode_one_shard(shard_texts):
    """Encode one shard with adaptive GPU batches; fall back to CPU on OOM."""
    if torch.cuda.is_available():
        for bs in GPU_BATCH_SIZES:
            try:
                gc.collect()
                torch.cuda.empty_cache()
                free_bytes, _ = torch.cuda.mem_get_info()
                print(f"    Trying GPU batch_size={bs} | free={free_bytes/(1024**3):.2f} GiB")
                arr = embedder.encode(
                    shard_texts,
                    show_progress_bar=True,
                    convert_to_numpy=True,
                    normalize_embeddings=True,
                    batch_size=bs,
                )
                return np.asarray(arr, dtype=np.float32)
            except torch.cuda.OutOfMemoryError:
                print(f"    ⚠️ GPU OOM at batch_size={bs}; retrying smaller batch...")
                gc.collect()
                torch.cuda.empty_cache()

        print("    ⚠️ GPU still out of memory. Encoding this shard on CPU...")
        gc.collect()
        torch.cuda.empty_cache()
        cpu_embedder = SentenceTransformer(EMBED_MODEL, device='cpu')
        try:
            arr = cpu_embedder.encode(
                shard_texts,
                show_progress_bar=True,
                convert_to_numpy=True,
                normalize_embeddings=True,
                batch_size=CPU_BATCH_SIZE,
            )
            return np.asarray(arr, dtype=np.float32)
        finally:
            del cpu_embedder
            gc.collect()
    else:
        arr = embedder.encode(
            shard_texts,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
            batch_size=CPU_BATCH_SIZE,
        )
        return np.asarray(arr, dtype=np.float32)


# Encode and save each shard immediately. Nothing is accumulated in RAM.
shard_paths = []
num_shards = (total + SUB_BATCH_SIZE - 1) // SUB_BATCH_SIZE

for shard_num, start in enumerate(range(0, total, SUB_BATCH_SIZE)):
    end = min(start + SUB_BATCH_SIZE, total)
    shard_texts = texts[start:end]

    print("\n" + "-"*80)
    print(f"Shard {shard_num + 1}/{num_shards}: rows {start:,} to {end-1:,} ({len(shard_texts):,} chunks)")

    shard_emb = encode_one_shard(shard_texts)

    if shard_emb.shape[0] != len(shard_texts):
        raise RuntimeError(
            f"Shard row mismatch: got {shard_emb.shape[0]}, expected {len(shard_texts)}"
        )
    if shard_emb.shape[1] != embedding_dim:
        raise RuntimeError(
            f"Embedding dimension mismatch: got {shard_emb.shape[1]}, expected {embedding_dim}"
        )

    shard_path = os.path.join(SHARD_DIR, f'shard_{shard_num:04d}.npy')
    np.save(shard_path, shard_emb)
    shard_paths.append(shard_path)
    print(f"✅ Saved shard: {shard_path}  |  shape={shard_emb.shape}")

    del shard_emb, shard_texts
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n✅ All embedding shards created.")

# Free GPU model before assembling/indexing.
del embedder
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# -----------------------------------------------------------------------
# ASSEMBLE SHARDS -> embeddings.npy WITHOUT LOADING ALL SHARDS INTO RAM
# -----------------------------------------------------------------------
print("\n" + "="*80)
print("ASSEMBLING FINAL EMBEDDINGS")
print("="*80)

emb_path = os.path.join(SAVE_PATH, 'embeddings.npy')

# open_memmap creates a normal .npy file while allowing incremental writes.
embeddings = np.lib.format.open_memmap(
    emb_path,
    mode='w+',
    dtype=np.float32,
    shape=(total, embedding_dim),
)

write_pos = 0
for i, shard_path in enumerate(shard_paths, start=1):
    shard_arr = np.load(shard_path, mmap_mode='r')
    rows = shard_arr.shape[0]
    embeddings[write_pos:write_pos + rows] = shard_arr
    write_pos += rows
    print(f"  ✅ Assembled shard {i}/{len(shard_paths)} | rows written: {write_pos:,}/{total:,}")
    del shard_arr
    gc.collect()

embeddings.flush()

assert write_pos == total, f"Shape mismatch: {write_pos} != {total}"
assert embeddings.shape == (total, embedding_dim)

print(f"✅ Embeddings shape: {embeddings.shape}")
print(f"✅ Saved: {emb_path}")


# -----------------------------------------------------------------------
# 19) BUILD FAISS INDEX  (incremental, RAM-safer)
# -----------------------------------------------------------------------
print("\n" + "="*80)
print("BUILDING FAISS INDEX")
print("="*80)

dimension = embedding_dim
faiss_index = faiss.IndexFlatIP(dimension)

# Add directly from saved shards; no full in-RAM copy is required.
for i, shard_path in enumerate(shard_paths, start=1):
    shard_arr = np.load(shard_path, mmap_mode='r')
    shard_float32 = np.ascontiguousarray(shard_arr, dtype=np.float32)
    faiss_index.add(shard_float32)
    print(f"  ✅ FAISS shard {i}/{len(shard_paths)} | total vectors: {faiss_index.ntotal:,}")
    del shard_arr, shard_float32
    gc.collect()

assert faiss_index.ntotal == total, \
    f"FAISS mismatch: {faiss_index.ntotal} != {total}"

faiss_path = os.path.join(SAVE_PATH, 'faiss_index.index')
faiss.write_index(faiss_index, faiss_path)
print(f'✅ FAISS: {faiss_index.ntotal:,} vectors  |  dim={dimension}')
print(f'✅ Saved: {faiss_path}')

# Shards are no longer needed after embeddings.npy + FAISS are complete.
shutil.rmtree(SHARD_DIR, ignore_errors=True)
print('✅ Temporary embedding shards cleaned up.')


# -----------------------------------------------------------------------
# 20) BUILD BM25 INDEX  (same improved tokenizer as original)
# -----------------------------------------------------------------------
print("\n" + "="*80)
print("BUILDING BM25 INDEX  (improved tokenizer)")
print("="*80)

# STOPWORDS and tokenize_for_bm25 were already defined in Section 2.
# Reuse them exactly so the data/tokenization logic is unchanged.
tokenized = [
    tokenize_for_bm25(doc)
    for doc in tqdm(df['content'].tolist(), desc='Tokenizing', leave=False)
]

bm25 = BM25Okapi(tokenized)

bm25_path = os.path.join(SAVE_PATH, 'bm25_index.pkl')
with open(bm25_path, 'wb') as f:
    pickle.dump(bm25, f, protocol=pickle.HIGHEST_PROTOCOL)

with open(os.path.join(SAVE_PATH, 'bm25_tokenizer.pkl'), 'wb') as f:
    pickle.dump('improved', f)

print(f'✅ BM25: {bm25.corpus_size:,} documents')
print(f'✅ Saved: {bm25_path}')

# tokenized is not needed after bm25 is built.
del tokenized
gc.collect()


# -----------------------------------------------------------------------
# 21) SAVE METADATA
# -----------------------------------------------------------------------
print("\n" + "="*80)
print("SAVING METADATA")
print("="*80)

df_save = df.reset_index(drop=True)
chunks_path = os.path.join(SAVE_PATH, 'chunks_df.pkl')
df_save.to_pickle(chunks_path)
print(f'✅ chunks_df.pkl saved  |  shape: {df_save.shape}')


# -----------------------------------------------------------------------
# VERIFY ALIGNMENT
# -----------------------------------------------------------------------
print("\n" + "="*80)
print("VERIFYING KNOWLEDGE BASE ALIGNMENT")
print("="*80)

print(f"DataFrame rows : {len(df_save):,}")
print(f"Embedding rows : {embeddings.shape[0]:,}")
print(f"FAISS vectors  : {faiss_index.ntotal:,}")
print(f"BM25 documents : {bm25.corpus_size:,}")

assert len(df_save) == total
assert embeddings.shape[0] == total
assert faiss_index.ntotal == total
assert bm25.corpus_size == total

print(f"✅ All components contain the same {total:,} chunks in the same row order.")


# -----------------------------------------------------------------------
# 22) FINAL SUMMARY
# -----------------------------------------------------------------------
print("\n" + "="*80)
print("🎉  KNOWLEDGE BASE BUILD COMPLETE!")
print("="*80)

pdf_sources     = [s for s in df['source'].unique() if str(s).startswith('pdf:')]
pdf_chunk_count = len(df[df['source'].astype(str).str.startswith('pdf:')])
wiki_count      = len(df[df['source'] == 'wikipedia'])
pmc_count       = len(df[df['source'] == 'pmc_fulltext'])

# Actual file sizes instead of rough estimates.
def file_size_gb(path):
    return os.path.getsize(path) / (1024**3) if os.path.exists(path) else 0.0

print(f"""
📊 Knowledge Base Statistics
   ├─ Total chunks      : {len(df):,}
   ├─ Categories        : {df['category'].nunique()}  {list(df['category'].unique())}
   ├─ Unique sources    : {df['source'].nunique()}
   ├─ Embedding dim     : {embeddings.shape[1]}
   ├─ PDF files         : {len(pdf_sources)} files → {pdf_chunk_count:,} chunks
   ├─ Wikipedia         : {wiki_count:,} chunks
   ├─ PMC full-text     : {pmc_count:,} chunks
   └─ BM25 tokenizer    : IMPROVED (stopwords removed + punctuation stripped)

📁 Files saved to: {SAVE_PATH}/
   ├─ embeddings.npy         ({file_size_gb(emb_path):.2f} GB)
   ├─ faiss_index.index      ({file_size_gb(faiss_path):.2f} GB)
   ├─ bm25_index.pkl         ({file_size_gb(bm25_path):.2f} GB)
   ├─ bm25_tokenizer.pkl
   └─ chunks_df.pkl          ({file_size_gb(chunks_path):.2f} GB)

✅ Alignment verified:
   df row i == embeddings row i == FAISS vector i == BM25 document i
""")

print("="*80)
print("\n💾  EXAMPLE USAGE IN YOUR RAG SYSTEM:")
print("""
import pandas as pd, faiss, pickle, numpy as np
from sentence_transformers import SentenceTransformer
import re

PATH = 'KB_complete'

# Load
embeddings  = np.load(f'{PATH}/embeddings.npy', mmap_mode='r')
faiss_index = faiss.read_index(f'{PATH}/faiss_index.index')
with open(f'{PATH}/bm25_index.pkl', 'rb') as f:
    bm25 = pickle.load(f)
df = pd.read_pickle(f'{PATH}/chunks_df.pkl')

STOPWORDS = frozenset({'the','a','an','is','are','was','were','be','been',
    'have','has','had','do','does','did','will','would','and','but','or',
    'not','no','of','in','on','at','to','for','with','by','from','it','its'})

def tokenize(text):
    tokens = re.findall(r'[a-z0-9]+', text.lower())
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]

embedder = SentenceTransformer('BAAI/bge-base-en-v1.5')

def search(query, k=5):
    # Dense (FAISS)
    q_emb = embedder.encode(query, normalize_embeddings=True).reshape(1, -1)
    _, faiss_ids = faiss_index.search(q_emb, k)

    # Sparse (BM25)
    bm25_scores = bm25.get_scores(tokenize(query))
    bm25_ids = bm25_scores.argsort()[::-1][:k]

    # Merge & deduplicate
    all_ids = list(dict.fromkeys(list(faiss_ids[0]) + list(bm25_ids)))
    return df.iloc[all_ids][['category', 'source', 'content']].head(k)

print(search('What is aphasia?'))
""")
print("="*80)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 44.3 MB/s eta 0:00:00
Mounted at /content/drive
PDF folder: /content/drive/MyDrive/PDF_Sources
Folder exists: True

Select your PubMed (.txt) files and PDF textbooks from your computer:


Saving pubmed-biomedical-set (1).txt to pubmed-biomedical-set (1).txt
Saving pubmed-ClinicalLa-set.txt to pubmed-ClinicalLa-set.txt
Saving pubmed-Occupation-set.txt to pubmed-Occupation-set.txt
Saving pubmed-ClinicalPs-set.txt to pubmed-ClinicalPs-set.txt
Saving pubmed-speechlang-set.txt to pubmed-speechlang-set.txt
✅ PubMed/text file available at: /content/pubmed-biomedical-set (1).txt
✅ PubMed/text file available at: /content/pubmed-ClinicalLa-set.txt
✅ PubMed/text file available at: /content/pubmed-Occupation-set.txt
✅ PubMed/text file available at: /content/pubmed-ClinicalPs-set.txt
✅ PubMed/text file available at: /content/pubmed-speechlang-set.txt

PDFs currently in PDF_Sources:
  - Book_19.pdf
  - Book_23.pdf
  - Book_25.pdf
  - book 8.pdf
  - book1.pdf
  - book4.pdf
  - book_14 .pdf
  - book_15.pdf
  - book_17.pdf
  - book_18.pdf
  - celf-5-clinical-evaluation-of-language-fundamentals-5th-edition_compress.pdf
Total PDFs: 11
Save path : /content/drive/MyDrive/KB_complete
PDF fol

  biomedical_engineer                         +181 chunks


  clinical_laboratory_scientist               +120 chunks


  clinical_psychologist                       +123 chunks


...

Batches:   0%|          | 0/782 [00:00<?, ?it/s]

✅ Saved shard: /content/emb_shards/shard_0004.npy  |  shape=(50000, 768)

--------------------------------------------------------------------------------
Shard 6/6: rows 250,000 to 282,164 (32,165 chunks)
    Trying GPU batch_size=64 | free=14.00 GiB


Batches:   0%|          | 0/503 [00:00<?, ?it/s]

✅ Saved shard: /content/emb_shards/shard_0005.npy  |  shape=(32165, 768)

✅ All embedding shards created.

ASSEMBLING FINAL EMBEDDINGS
  ✅ Assembled shard 1/6 | rows written: 50,000/282,165
  ✅ Assembled shard 2/6 | rows written: 100,000/282,165
  ✅ Assembled shard 3/6 | rows written: 150,000/282,165
  ✅ Assembled shard 4/6 | rows written: 200,000/282,165
  ✅ Assembled shard 5/6 | rows written: 250,000/282,165
  ✅ Assembled shard 6/6 | rows written: 282,165/282,165
✅ Embeddings shape: (282165, 768)
✅ Saved: /content/drive/MyDrive/KB_complete/embeddings.npy

BUILDING FAISS INDEX
  ✅ FAISS shard 1/6 | total vectors: 50,000
  ✅ FAISS shard 2/6 | total vectors: 100,000
  ✅ FAISS shard 3/6 | total vectors: 150,000
  ✅ FAISS shard 4/6 | total vectors: 200,000
  ✅ FAISS shard 5/6 | total vectors: 250,000
  ✅ FAISS shard 6/6 | total vectors: 282,165
✅ FAISS: 282,165 vectors  |  dim=768
✅ Saved: /content/drive/MyDrive/KB_complete/faiss_index.index
✅ Temporary embedding shards cleaned up.

BUILD

✅ BM25: 282,165 documents
✅ Saved: /content/drive/MyDrive/KB_complete/bm25_index.pkl

SAVING METADATA
✅ chunks_df.pkl saved  |  shape: (282165, 3)

VERIFYING KNOWLEDGE BASE ALIGNMENT
DataFrame rows : 282,165
Embedding rows : 282,165
FAISS vectors  : 282,165
BM25 documents : 282,165
✅ All components contain the same 282,165 chunks in the same row order.

🎉  KNOWLEDGE BASE BUILD COMPLETE!

📊 Knowledge Base Statistics
   ├─ Total chunks      : 282,165
   ├─ Categories        : 6  ['clinical_laboratory_scientist', 'clinical_psychologist', 'occupational_therapist', 'biomedical_engineer', 'speech_pathologist', 'general_medical']
   ├─ Unique sources    : 21
   ├─ Embedding dim     : 768
   ├─ PDF files         : 10 files → 12,696 chunks
   ├─ Wikipedia         : 1,798 chunks
   ├─ PMC full-text     : 12,143 chunks
   └─ BM25 tokenizer    : IMPROVED (stopwords removed + punctuation stripped)

📁 Files saved to: /content/drive/MyDrive/KB_complete/
   ├─ embeddings.npy         (0.81 GB)
   ├─ fa